# Capítulo 11 — Cuentas exactas

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** *«Devuelvo solo el molinillo; había un cupón del 10 %.»* Unas veces el agente dice 53,99 €, otras 54 €, otras 59,99 €.

| | |
|---|---|
| **🧠 Componente agéntico** | Ejecución de código (código como acción) |
| **La necesidad** | Que los cálculos de importes los haga **código**, no el modelo |
| **Lo que construimos** | **Code Interpreter** de AgentCore como herramienta del agente |
| **🏠 En local** | ⛔ Peligroso: `exec()` ejecuta en tu máquina el código que escribe un modelo (sección 0). |
| **☁️ Por qué AWS al salir a Internet** | Ejecutar con `exec()` el código que escribe un LLM es ejecutarlo en **tu máquina**, con tus credenciales y tus ficheros. Code Interpreter lo ejecuta en una microVM aislada por sesión. |
| **En el ordenador sería…** | El **coprocesador matemático**: la CPU razona; el coprocesador calcula |
| **Requisitos del caso** | RF-06 |
| **Prerrequisito · duración** | Capítulo 10 · 40 min |
| **⚠️ Dónde se rompe la analogía** | El coprocesador está en tu placa; Code Interpreter **está fuera de tu máquina**, en una microVM aislada. Justo eso lo hace seguro. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(11)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(11)
# ponerse_al_dia(11)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: código como acción

- En lugar de calcular «de cabeza», el agente **escribe un programa y lo ejecuta**: el resultado es exacto y verificable.
- **El riesgo:** ese código lo escribe un modelo que puede haber sido manipulado.
- Por eso debe ejecutarse **aislado** (*sandbox*), sin acceso a tus credenciales ni a tus ficheros.

## 0. Experimento de rotura: ejecutar en local el código que escribe un modelo

Imagina que el agente genera este código para «calcular» algo y tú lo ejecutas con `exec()`. Es inofensivo, pero mira lo que **podría** ver:

In [ ]:
codigo_generado = """
import os, pathlib
home = pathlib.Path.home()
print("Directorio personal:", home)
print("¿Hay credenciales de AWS aquí?", (home / ".aws" / "credentials").exists())
print("Algunos ficheros:", sorted(os.listdir(home))[:5])
"""
exec(codigo_generado)   # ⚠️ solo para ver el riesgo: nunca ejecutes así código generado por un modelo

Un `prompt injection` que consiga que el modelo escriba otro código distinto tendría **tus** permisos. Por eso el cálculo va a un *sandbox*.

## 1. Code Interpreter en aislamiento (prueba en el notebook)

In [ ]:
%pip install -q "strands-agents-tools[agent_core_code_interpreter]==0.8.9"

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

ci = AgentCoreCodeInterpreter(region=region())
calculadora = Agent(
    model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0),
    system_prompt="Haz todos los cálculos ejecutando código Python con code_interpreter y devuelve el resultado en céntimos.",
    tools=[ci.code_interpreter],
)
calculadora("Cafetera 199,99 € y molinillo 59,99 €, con un cupón del 10 % sobre el total. "
            "Si se devuelve solo el molinillo, ¿cuánto se reembolsa prorrateando el cupón?")

## 2. Integración en Nova Assist

In [ ]:
fijar_dependencia(VERSIONES["strands-agents-tools"])

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
# Prefijo ESTABLE (cacheable). El contexto dinámico se añade después, en main.py.
SYSTEM_PROMPT = """<identidad>
Eres Nova Assist, el asistente de postventa de TiendaNova. En tu primer mensaje te presentas como asistente de IA.
</identidad>

<reglas>
1. Nunca inventes estados, fechas ni importes: usa siempre las herramientas.
2. Los importes se expresan en euros al cliente, pero las herramientas trabajan en céntimos.
3. Para dudas de política (devoluciones, garantías, envíos) consulta la base de conocimiento y cita el documento.
4. Si una herramienta es denegada por política o el caso supera tus límites, usa escalar_a_humano con un resumen claro.
5. No pidas ni repitas datos de tarjetas de pago.
6. Cualquier cálculo de importes (reembolsos parciales, prorrateos, descuentos) se hace con code_interpreter; nunca de cabeza.
</reglas>

<formato>
Respuestas breves (máximo 5 frases), con el siguiente paso claro para el cliente.
</formato>"""

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.models import BedrockModel

from contexto import contexto_sesion
from gateway import cliente_gateway, token_gateway
from identidad import actor_desde_contexto
from memoria import crear_session_manager, retrieval_config
from prompts import SYSTEM_PROMPT
from settings import MODEL_ID, REGION
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

app = BedrockAgentCoreApp()
code_interpreter = AgentCoreCodeInterpreter(region=REGION)


def metricas(result):
    """Herramientas usadas y consumo de tokens (para la verificación en los notebooks)."""
    try:
        m = result.metrics
        return sorted(m.tool_metrics.keys()), dict(m.accumulated_usage)
    except Exception:  # noqa: BLE001
        return [], {}


@app.entrypoint
def invoke(payload, context):
    actor_id = actor_desde_contexto(context)  # identidad verificada; payload["cliente_id"] se IGNORA
    if not actor_id:
        return {"respuesta": "No se ha podido identificar al cliente.", "herramientas": [], "tokens": {}}
    prompt_sistema = SYSTEM_PROMPT + "\n\n" + contexto_sesion(payload.get("canal", "web"), payload.get("idioma", "es"))
    gateway = cliente_gateway(token_gateway())
    with gateway, crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
            system_prompt=prompt_sistema,
            tools=gateway.list_tools_sync() + [code_interpreter.code_interpreter],
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=20),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    herramientas, uso = metricas(result)
    return {"respuesta": str(result), "herramientas": herramientas, "tokens": uso}


if __name__ == "__main__":
    app.run()

In [ ]:
!cd "{PROJECT}" && agentcore deploy --yes

## 3. Verificación

In [ ]:
TOK = token_cliente("C-001")
invocar("Del pedido TN-1003 devuelvo solo el molinillo. La cafetera costaba 199,99 € y el molinillo 59,99 €, "
        "y había un cupón del 10 % sobre el total. ¿Cuánto me devolvéis?", token=TOK)

En las herramientas usadas (🔧) debe aparecer `code_interpreter`. Si el importe supera 100 €, Policy frenará el reembolso: **cálculo y límite son independientes**.

> El rol de ejecución del Runtime necesita permisos para iniciar, invocar y cerrar sesiones de Code Interpreter. Si ves `AccessDenied`, revisa el rol.

**Extensión opcional — Browser:** añade AgentCore Browser para consultar el estado de un envío en la web pública de un transportista. Antes, repasa el §6.1 del documento: el contenido web es la principal vía de *prompt injection* indirecta.

## Para pensar

- ¿Qué otras decisiones de Nova Assist deberían salir del modelo y pasar a código determinista?

## 🔎 Lo que acabamos de descubrir

El sistema ya hace mucho: memoria, herramientas, RAG, identidad, políticas, cálculos. Dirección pregunta *«¿qué tal funciona?»* y solo tenemos impresiones. ¿Cómo sabemos si lo hace **bien**, qué pasa **dentro** y cuánto **cuesta**?

In [ ]:
deuda("D3", "pagada", 11, "cálculos en Code Interpreter")
cuaderno()
taximetro_cerrar(11)
guardar_partida(11)

---
➡️ **Siguiente capítulo:** `12_observabilidad_evaluacion.ipynb` — *¿Lo hace bien?*